In [2]:

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data.grd_parser import (
    read_grd_file,
    get_grid_coordinates,
    grd_to_dataframe
)

In [6]:
from src.data.grd_parser import load_multiple_years
import pandas as pd 

In [8]:
years = range(2010, 2026)
DATA_DIR = Path("../Data")
file_stats = []

for year in years:
    file_path = DATA_DIR / f"{year}.GRD"

    data = read_grd_file(file_path)

    file_stats.append({
        "year": year,
        "file_size_mb": file_path.stat().st_size / (1024 ** 2),
        "days": data.shape[0],
        "grid_rows": data.shape[1],
        "grid_cols": data.shape[2],
        "dtype": str(data.dtype)
    })

file_stats_df = pd.DataFrame(file_stats)

file_stats_df

,year,file_size_mb,days,grid_rows,grid_cols,dtype
0,2010,1.338062,365,31,31,float32
1,2011,1.338062,365,31,31,float32
2,2012,1.341728,366,31,31,float32
3,2013,1.338062,365,31,31,float32
4,2014,1.338062,365,31,31,float32
5,2015,1.338062,365,31,31,float32
6,2016,1.341728,366,31,31,float32
7,2017,1.338062,365,31,31,float32
8,2018,1.338062,365,31,31,float32
9,2019,1.338062,365,31,31,float32


In [9]:
df_all = load_multiple_years(
    DATA_DIR,
    start_year=2010,
    end_year=2025
)

Processing 2010...
  Days: 365 | Rows: 350,765
Processing 2011...
  Days: 365 | Rows: 350,765
Processing 2012...
  Days: 366 | Rows: 351,726
Processing 2013...
  Days: 365 | Rows: 350,765
Processing 2014...
  Days: 365 | Rows: 350,765
Processing 2015...
  Days: 365 | Rows: 350,765
Processing 2016...
  Days: 366 | Rows: 351,726
Processing 2017...
  Days: 365 | Rows: 350,765
Processing 2018...
  Days: 365 | Rows: 350,765
Processing 2019...
  Days: 365 | Rows: 350,765
Processing 2020...
  Days: 366 | Rows: 351,726
Processing 2021...
  Days: 365 | Rows: 350,765
Processing 2022...
  Days: 365 | Rows: 350,765
Processing 2023...
  Days: 365 | Rows: 350,765
Processing 2024...
  Days: 366 | Rows: 351,726
Processing 2025...
  Days: 365 | Rows: 350,765


In [10]:
location_stats = (
    df_all
    .groupby(["latitude", "longitude"])["max_temperature"]
    .agg(
        valid_days="count",
        missing_days=lambda x: x.isna().sum()
    )
    .reset_index()
)

In [11]:
valid_locations = location_stats[
    location_stats["valid_days"] > 0
][["latitude", "longitude"]]

In [12]:
location_stats = (
    df_all
    .groupby(["latitude", "longitude"])["max_temperature"]
    .agg(
        valid_days="count",
        missing_days=lambda x: x.isna().sum()
    )
    .reset_index()
)

location_stats["total_days"] = (
    location_stats["valid_days"] +
    location_stats["missing_days"]
)

location_stats["missing_percentage"] = (
    location_stats["missing_days"]
    / location_stats["total_days"]
    * 100
)

location_stats.head()

,latitude,longitude,valid_days,missing_days,total_days,missing_percentage
0,7.5,67.5,0,5844,5844,100.0
1,7.5,68.5,0,5844,5844,100.0
2,7.5,69.5,0,5844,5844,100.0
3,7.5,70.5,0,5844,5844,100.0
4,7.5,71.5,0,5844,5844,100.0


In [13]:
print("Total grid points:", len(location_stats))

print(
    "\nAlways invalid:",
    (location_stats["valid_days"] == 0).sum()
)

print(
    "At least one valid observation:",
    (location_stats["valid_days"] > 0).sum()
)

print(
    "\nIntermittently missing:",
    (
        (location_stats["valid_days"] > 0) &
        (location_stats["missing_days"] > 0)
    ).sum()
)

Total grid points: 961

Always invalid: 606
At least one valid observation: 355

Intermittently missing: 39


In [14]:
valid_locations = location_stats[
    location_stats["valid_days"] > 0
][["latitude", "longitude"]].copy()

print("Valid spatial locations:", len(valid_locations))

Valid spatial locations: 355


In [15]:
df_clean = df_all.merge(
    valid_locations,
    on=["latitude", "longitude"],
    how="inner"
)

print("Original shape:", df_all.shape)
print("Clean shape:", df_clean.shape)

Original shape: (5616084, 4)
Clean shape: (2074620, 4)


In [16]:
missing_summary = (
    df_clean["max_temperature"]
    .isna()
    .sum()
)

print("Remaining missing temperature values:", missing_summary)

print(
    "Missing percentage:",
    round(
        df_clean["max_temperature"].isna().mean() * 100,
        4
    ),
    "%"
)

Remaining missing temperature values: 4244
Missing percentage: 0.2046 %


In [17]:
missing_rows = df_clean[
    df_clean["max_temperature"].isna()
]

missing_rows.head(20)

,date,latitude,longitude,max_temperature
308,2010-01-01,29.5,95.5,NaN
309,2010-01-01,29.5,96.5,NaN
663,2010-01-02,29.5,95.5,NaN
664,2010-01-02,29.5,96.5,NaN
1018,2010-01-03,29.5,95.5,NaN
1019,2010-01-03,29.5,96.5,NaN
1059,2010-01-03,35.5,78.5,NaN
1060,2010-01-03,35.5,79.5,NaN
1061,2010-01-03,36.5,72.5,NaN
1062,2010-01-03,36.5,73.5,NaN


In [18]:
missing_rows.groupby(
    ["latitude", "longitude"]
).size().sort_values(
    ascending=False
).head(20)

latitude  longitude
29.5      96.5         865
          95.5         865
35.5      79.5         605
36.5      74.5         444
          73.5         444
          72.5         444
35.5      78.5          98
36.5      75.5          98
29.5      94.5          56
27.5      96.5          52
          97.5          52
28.5      96.5          52
35.5      75.5          16
34.5      73.5          16
35.5      74.5          16
          73.5          16
28.5      95.5          15
27.5      94.5          10
          95.5          10
28.5      94.5          10
dtype: int64

In [19]:
print("Date range:")
print(df_clean["date"].min(), "→", df_clean["date"].max())

print("\nUnique dates:", df_clean["date"].nunique())

print(
    "\nUnique locations:",
    df_clean[["latitude", "longitude"]]
    .drop_duplicates()
    .shape[0]
)

print("\nTemperature statistics:")
print(df_clean["max_temperature"].describe())

Date range:
2010-01-01 00:00:00 → 2025-12-31 00:00:00

Unique dates: 5844

Unique locations: 355

Temperature statistics:
count    2.070376e+06
mean     3.089599e+01
std      5.884315e+00
min      2.015030e+00
25%      2.814436e+01
50%      3.130458e+01
75%      3.438000e+01
max      4.877099e+01
Name: max_temperature, dtype: float64


In [20]:
processed_dir = PROJECT_ROOT / "Data" / "processed"
processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    processed_dir /
    "imd_temperature_2010_2025.parquet"
)

df_clean.to_parquet(
    output_path,
    engine="pyarrow",
    index=False
)

print(f"Saved to: {output_path}")
print(
    f"File size: "
    f"{output_path.stat().st_size / (1024 ** 2):.2f} MB"
)

Saved to: c:\Users\Qudsiya Siddique\Desktop\Climat\Data\processed\imd_temperature_2010_2025.parquet
File size: 8.09 MB
